# 🛰️ Notebook 2 — RT-DETR-L + YOLOv8m · Training + 3-Slice SAHI
## Satellite Aerial Object Detection — Conference Paper Pipeline
### ⚠️ This notebook is FULLY SELF-CONTAINED. No dependency on Notebook 1 session.

**Cell order:**
1. **Cell 1** — Setup, device check, all shared config + helpers
2. **Cell 2** — Dataset preprocessing (COCO → YOLO labels + data.yaml)
3. **Cell 3** — Train RT-DETR-L + 3-slice SAHI
4. **Cell 4** — Train YOLOv8m + 3-slice SAHI + export combined CSV

> **GPU strongly recommended.** Auto-falls back to CPU (slow).  
> batch=4 for both models — safe for all hardware including 16 GB VRAM.

## 📦 Cell 1 — Setup · Device · Shared Config · All Helpers

In [1]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1 — SETUP + DEVICE + SHARED CONFIG + HELPERS
# ═══════════════════════════════════════════════════════════════════════

# ── 1A. Install ─────────────────────────────────────────────────────
!pip install ultralytics sahi seaborn scipy -q

import os, json, shutil, time, gc, warnings
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

# ── 1B. Device detection (GPU → CPU fallback) ────────────────────────
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("=" * 65)
print("DEVICE CONFIGURATION")
print("=" * 65)
if torch.cuda.is_available():
    print(f"  ✅ GPU : {torch.cuda.get_device_name(0)}")
    print(f"     VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print(f"     CUDA: {torch.version.cuda}")
else:
    print("  ⚠️  No GPU detected — CPU mode active.")
    print("      Training will work but be significantly slower.")
    print("      → Kaggle: Settings → Accelerator → GPU T4 x2 → Save → Restart")
print(f"  Active device: {device}")

# ── 1C. Paths ────────────────────────────────────────────────────────
DATASET_PATH = "/kaggle/input/datasets/rifat4018/satelite-coco"
WORK_DIR     = "/kaggle/working"
SAVE_DIR     = f"{WORK_DIR}/saved_models"
YOLO_DIR     = f"{WORK_DIR}/dataset"
YAML_PATH    = f"{WORK_DIR}/data.yaml"
PLOT_DIR     = f"{WORK_DIR}/plots"

for d in [SAVE_DIR, YOLO_DIR, PLOT_DIR]:
    os.makedirs(d, exist_ok=True)

TRAIN_JSON = f"{DATASET_PATH}/train/_annotations.coco.json"
VAL_JSON   = f"{DATASET_PATH}/valid/_annotations.coco.json"
TEST_JSON  = f"{DATASET_PATH}/test/_annotations.coco.json"
TEST_IMG_DIR = f"{YOLO_DIR}/test/images"

# ════════════════════════════════════════════════════════════════════════
# THE 15 CANONICAL HYPERPARAMETERS
# Exact copy from NB1 Cell 2. Must match byte-for-byte.
# NB3 reads hp_* columns from the CSV and will flag any mismatch.
# ════════════════════════════════════════════════════════════════════════
TRACKED_PARAM_KEYS = [
    "epochs", "imgsz", "batch", "patience", "warmup_epochs",   # P01–P05
    "mosaic", "mixup", "copy_paste", "degrees", "translate",   # P06–P10
    "scale",  "flipud", "fliplr", "hsv_h", "hsv_s",           # P11–P15
]

COMMON_TRAIN_PARAMS = dict(
    epochs        =  50,    # P01  — match NB1 (set 100 for final paper run)
    imgsz         = 640,    # P02
    batch         =   4,    # P03  default; overridden per model below
    patience      =  20,    # P04
    warmup_epochs =   3,    # P05
    mosaic        = 1.0,    # P06
    mixup         = 0.1,    # P07
    copy_paste    = 0.1,    # P08
    degrees       = 45.0,   # P09
    translate     = 0.1,    # P10
    scale         = 0.5,    # P11
    flipud        = 0.5,    # P12
    fliplr        = 0.5,    # P13
    hsv_h         = 0.015,  # P14
    hsv_s         = 0.7,    # P15
    # ── Fixed (not in tracked 15) ─────────────────────────────────
    data          = YAML_PATH,
    workers       = 2,
    seed          = 42,
    verbose       = False,
)

# ── 3-Slice SAHI config — IDENTICAL to NB1 ──────────────────────────
SAHI_SIZES   = [480, 320, 160]   # SP01
SAHI_OVERLAP = 0.2               # SP02
SAHI_CONF    = 0.25              # SP03
SAHI_N_IMGS  = 50                # SP04

# Print param table for verification
descriptions = {
    "epochs":"Total training epochs","imgsz":"Input image size (px)",
    "batch":"Mini-batch size (model-specific)","patience":"Early-stop patience (epochs)",
    "warmup_epochs":"LR warm-up period (epochs)","mosaic":"Mosaic aug probability",
    "mixup":"Mixup aug probability","copy_paste":"Copy-paste aug probability",
    "degrees":"Random rotation range (±deg)","translate":"Random translation (fraction)",
    "scale":"Random scale range (fraction)","flipud":"Vertical flip probability",
    "fliplr":"Horizontal flip probability","hsv_h":"Hue jitter (fraction)",
    "hsv_s":"Saturation jitter",
}
print("\n" + "="*65)
print("THE 15 CANONICAL TRAINING HYPERPARAMETERS (must match NB1)")
print("="*65)
print(f"  {'ID':<5} {'Parameter':<18} {'Value':<8}  Description")
print("  " + "─"*60)
for i, k in enumerate(TRACKED_PARAM_KEYS, 1):
    print(f"  P{i:02d}   {k:<18} {str(COMMON_TRAIN_PARAMS[k]):<8}  {descriptions[k]}")
print(f"\n  SAHI slice sizes : {SAHI_SIZES}")
print(f"  SAHI overlap     : {SAHI_OVERLAP}")
print(f"  SAHI conf        : {SAHI_CONF}")
print(f"  batch this NB    : RT-DETR-L=4 | YOLOv8m=4")

# ════════════════════════════════════════════════════════════════════════
# SHARED HELPERS — used in Cell 3 and Cell 4
# ════════════════════════════════════════════════════════════════════════
from ultralytics import YOLO
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

def run_sahi_ablation(model, model_path, test_img_dir, slice_size):
    """
    Runs standard inference AND SAHI sliced inference on the same images.
    Returns std_avg_det, sahi_avg_det, sahi_gain_pct for one slice size.
    """
    imgs = [os.path.join(test_img_dir, f)
            for f in sorted(os.listdir(test_img_dir))
            if f.lower().endswith((".jpg", ".png"))][:SAHI_N_IMGS]

    if not imgs:
        return {
            "std_avg_det": 0, "sahi_avg_det": 0, "sahi_gain_pct": 0,
            "sahi_slice": slice_size, "sahi_overlap": SAHI_OVERLAP,
            "sahi_conf": SAHI_CONF, "sahi_n_imgs": 0,
        }

    # Standard (full-image) inference
    std_counts = []
    for img_path in imgs:
        res = model.predict(img_path, conf=SAHI_CONF, verbose=False, device=device)
        std_counts.append(len(res[0].boxes) if res and res[0].boxes is not None else 0)

    # SAHI sliced inference
    det_model = AutoDetectionModel.from_pretrained(
        model_type           = "yolov8",
        model_path           = model_path,
        confidence_threshold = SAHI_CONF,
        device               = device,
    )
    sahi_counts = []
    for img_path in imgs:
        r = get_sliced_prediction(
            img_path, det_model,
            slice_height         = slice_size,
            slice_width          = slice_size,
            overlap_height_ratio = SAHI_OVERLAP,
            overlap_width_ratio  = SAHI_OVERLAP,
            verbose              = 0,
        )
        sahi_counts.append(len(r.object_prediction_list))
    del det_model
    gc.collect()

    std_avg  = float(np.mean(std_counts))
    sahi_avg = float(np.mean(sahi_counts))
    return {
        "std_avg_det":   round(std_avg,  2),
        "sahi_avg_det":  round(sahi_avg, 2),
        "sahi_gain_pct": round(100 * (sahi_avg - std_avg) / max(std_avg, 1e-6), 1),
        "sahi_slice":    slice_size,
        "sahi_overlap":  SAHI_OVERLAP,
        "sahi_conf":     SAHI_CONF,
        "sahi_n_imgs":   len(imgs),
    }


def extract_metrics(results, base_name, train_time_sec, sahi_dict,
                    batch_used, tracked_params):
    """
    Loads best.pt, runs val(), measures FPS, builds the result row dict.
    Identical schema to NB1 so the CSVs can be concatenated directly.
    """
    best_pt  = os.path.join(results.save_dir, "weights", "best.pt")
    m        = YOLO(best_pt)
    val_res  = m.val(data=YAML_PATH, conf=SAHI_CONF, verbose=False, device=device)
    vd       = val_res.results_dict
    params_m = sum(p.numel() for p in m.model.parameters()) / 1e6

    # FPS benchmark (30 forward passes on device)
    dummy = torch.zeros(1, 3, 640, 640).to(device)
    t0    = time.time()
    with torch.no_grad():
        for _ in range(30):
            m.model(dummy)
    fps = round(30 / (time.time() - t0), 1)

    p = vd.get("metrics/precision(B)", 0)
    r = vd.get("metrics/recall(B)",    0)

    row = {
        # Identity
        "Model":         f"{base_name}_S{sahi_dict['sahi_slice']}",
        "base_model":    base_name,
        "sahi_slice":    sahi_dict["sahi_slice"],
        # Detection metrics (identical columns to NB1)
        "mAP50":         round(vd.get("metrics/mAP50(B)",    0), 4),
        "mAP50_95":      round(vd.get("metrics/mAP50-95(B)", 0), 4),
        "Precision":     round(p, 4),
        "Recall":        round(r, 4),
        "F1":            round(2 * p * r / max(p + r, 1e-6), 4),
        "AP50_plane":    round(float(val_res.box.ap50[0]) if hasattr(val_res.box, "ap50") else 0, 4),
        "AP50_ship":     round(float(val_res.box.ap50[1]) if hasattr(val_res.box, "ap50") else 0, 4),
        "AP50_vehicle":  round(float(val_res.box.ap50[2]) if hasattr(val_res.box, "ap50") else 0, 4),
        # Speed / size
        "FPS":           fps,
        "Params_M":      round(params_m, 2),
        "Train_time_hr": round(train_time_sec / 3600, 2),
        # SAHI ablation
        "std_avg_det":   sahi_dict["std_avg_det"],
        "sahi_avg_det":  sahi_dict["sahi_avg_det"],
        "sahi_gain_pct": sahi_dict["sahi_gain_pct"],
        "sahi_overlap":  sahi_dict["sahi_overlap"],
        # 15 tracked hyperparameters (hp_ prefix — NB3 reads these)
        **{f"hp_{k}": v for k, v in tracked_params.items()},
    }
    del m
    gc.collect()
    return row, best_pt

print("\n✅ Cell 1 complete — all helpers loaded.")
print("📌 Proceed to Cell 2 — Dataset Preprocessing")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 7.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
DEVICE CONFIGURATION
  ✅ GPU : Tesla T4
     VRAM: 15.6 GB
     CUDA: 12.8
  Active device: cuda:0

THE 15 CANONICAL TRAINING HYPERPARAMETERS (must match NB1)
  ID    Parameter          Value     Description
  ────────────────────────────────────────────────────────────
  P01   epochs             50        Total training epochs
  P02   imgsz              640       Input image size (px)
  P03   batch              4     

## ⚙️ Cell 2 — Dataset Preprocessing (COCO → YOLO + data.yaml)
> Fully self-contained — builds the dataset from scratch.  
> Safe to re-run: existing files are overwritten cleanly.

In [2]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 2 — DATASET PREPROCESSING
# COCO JSON → YOLO TXT labels (clipped + validated) + data.yaml
# ═══════════════════════════════════════════════════════════════════════

ID_MAP = {1: 0, 2: 1, 3: 2}   # plane=0, ship=1, vehicle=2

def convert_coco_to_yolo(json_path, img_src_dir, dest_img_dir, dest_lbl_dir):
    """
    Converts one COCO split to YOLO format.
    - Clips all bbox coordinates to [0, 1] (fixes negative/out-of-bound values)
    - Skips degenerate boxes (zero width or height after clipping)
    - Copies images alongside labels
    """
    os.makedirs(dest_img_dir, exist_ok=True)
    os.makedirs(dest_lbl_dir, exist_ok=True)

    with open(json_path) as f:
        coco = json.load(f)

    images     = {img["id"]: img for img in coco["images"]}
    ann_by_img = {}
    for ann in coco["annotations"]:
        ann_by_img.setdefault(ann["image_id"], []).append(ann)

    converted = skipped = invalid = 0
    for img_id, img_info in images.items():
        fname = img_info["file_name"]
        src   = os.path.join(img_src_dir, fname)
        if not os.path.exists(src):
            skipped += 1
            continue

        shutil.copy(src, os.path.join(dest_img_dir, fname))
        W, H  = img_info["width"], img_info["height"]
        lines = []

        for ann in ann_by_img.get(img_id, []):
            cid = ID_MAP.get(ann["category_id"])
            if cid is None:
                continue
            x, y, w, h = ann["bbox"]
            # Clip coordinates to valid [0, 1] range — fixes dataset errors
            cx = np.clip((x + w / 2) / W, 0.0, 1.0)
            cy = np.clip((y + h / 2) / H, 0.0, 1.0)
            nw = np.clip(w / W,            0.0, 1.0)
            nh = np.clip(h / H,            0.0, 1.0)
            if nw > 1e-4 and nh > 1e-4:
                lines.append(f"{cid} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
            else:
                invalid += 1

        lbl_name = os.path.splitext(fname)[0] + ".txt"
        with open(os.path.join(dest_lbl_dir, lbl_name), "w") as f:
            f.write("\n".join(lines))
        converted += 1

    print(f"  ✅ {converted:>5} converted | {skipped:>4} missing | {invalid:>4} degenerate removed")


print("=" * 65)
print("BUILDING YOLO DATASET (Notebook 2)")
print("=" * 65)

for split, json_f in [("train", TRAIN_JSON), ("val", VAL_JSON), ("test", TEST_JSON)]:
    src_dir = f"{DATASET_PATH}/{split if split != 'val' else 'valid'}"
    print(f"Converting {split}...")
    convert_coco_to_yolo(
        json_f, src_dir,
        f"{YOLO_DIR}/{split}/images",
        f"{YOLO_DIR}/{split}/labels",
    )

# Write data.yaml
with open(YAML_PATH, "w") as f:
    f.write(f"""train: {YOLO_DIR}/train/images
val:   {YOLO_DIR}/val/images
test:  {YOLO_DIR}/test/images

nc: 3
names: [plane, ship, vehicle]
""")

print(f"\n✅ data.yaml saved → {YAML_PATH}")
print("\n  Dataset structure:")
for split in ["train", "val", "test"]:
    img_dir = f"{YOLO_DIR}/{split}/images"
    lbl_dir = f"{YOLO_DIR}/{split}/labels"
    n_imgs = len([f for f in os.listdir(img_dir) if f.endswith(".jpg")]) if os.path.exists(img_dir) else 0
    n_lbls = len([f for f in os.listdir(lbl_dir) if f.endswith(".txt")]) if os.path.exists(lbl_dir) else 0
    print(f"  {split:<6}: {n_imgs:>5} images | {n_lbls:>5} label files")

print("\n✅ Preprocessing complete.")
print("📌 Proceed to Cell 3 — Train RT-DETR-L")


BUILDING YOLO DATASET (Notebook 2)
Converting train...
  ✅  1305 converted |    0 missing |    0 degenerate removed
Converting val...
  ✅    54 converted |    0 missing |    0 degenerate removed
Converting test...
  ✅    55 converted |    0 missing |    0 degenerate removed

✅ data.yaml saved → /kaggle/working/data.yaml

  Dataset structure:
  train :  1305 images |  1305 label files
  val   :    54 images |    54 label files
  test  :    55 images |    55 label files

✅ Preprocessing complete.
📌 Proceed to Cell 3 — Train RT-DETR-L


## 🤖 Cell 3 — Train RT-DETR-L + 3-Slice SAHI Ablation
> **Expected time:** ~2.5 hr on T4 GPU · ~8+ hr on CPU  
> RT-DETR-L is a transformer-based detector. batch=4 — safe for all hardware.  
> Produces 3 rows in CSV (one per SAHI slice size: 480 / 320 / 160 px).

In [3]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3 — TRAIN RT-DETR-L + 3-SLICE SAHI ABLATION
# ═══════════════════════════════════════════════════════════════════════

RTDETR_BATCH = 4   # ← conservative — prevents OOM on all hardware

print("=" * 65)
print(f"🏋️  Training RT-DETR-L  |  batch={RTDETR_BATCH}  |  device={device}")
print("=" * 65)
print("   Architecture : Transformer-based (DETReg) — strong on large objects")
print(f"  ⏱  Expected  : ~2.5 hr on T4 GPU | ~8+ hr on CPU")
print("   Do NOT interrupt — transformer training is sensitive to early stop")

# Build the tracked param row for this model
tracked_rtdetr = {k: COMMON_TRAIN_PARAMS[k] for k in TRACKED_PARAM_KEYS}
tracked_rtdetr["batch"] = RTDETR_BATCH   # override P03 with actual batch

# Build training call (strip keys that .train() doesn't accept)
train_call_rtdetr = {k: v for k, v in COMMON_TRAIN_PARAMS.items()
                     if k not in ("conf", "iou")}
train_call_rtdetr.update({
    "batch":   RTDETR_BATCH,
    "project": f"{WORK_DIR}/runs",
    "name":    "RT-DETR-L",
    "device":  0 if torch.cuda.is_available() else "cpu",
})

model_rtdetr   = YOLO("rtdetr-l.pt")
t_start        = time.time()
results_rtdetr = model_rtdetr.train(**train_call_rtdetr)
t_rtdetr       = time.time() - t_start
print(f"\n  ⏱  RT-DETR-L training complete: {t_rtdetr/3600:.2f} hr")

# ── 3-Slice SAHI ablation ────────────────────────────────────────────
best_rtdetr  = os.path.join(results_rtdetr.save_dir, "weights", "best.pt")
rows_rtdetr  = []

print("\n  🔪 Running 3-slice SAHI ablation on RT-DETR-L...")
print(f"  {'Slice':>8}  {'Standard':>10}  {'SAHI':>10}  {'Gain %':>8}")
print("  " + "─"*44)

for s_size in SAHI_SIZES:
    sahi_info = run_sahi_ablation(model_rtdetr, best_rtdetr, TEST_IMG_DIR, s_size)
    print(f"  {s_size:>6}px  {sahi_info['std_avg_det']:>10.2f}  "
          f"{sahi_info['sahi_avg_det']:>10.2f}  +{sahi_info['sahi_gain_pct']:>6.1f}%")
    row, _ = extract_metrics(results_rtdetr, "RT-DETR-L", t_rtdetr,
                             sahi_info, RTDETR_BATCH, tracked_rtdetr)
    rows_rtdetr.append(row)

# Save model weight
out_rtdetr = f"{SAVE_DIR}/RT-DETR-L_best.pt"
shutil.copy(best_rtdetr, out_rtdetr)
print(f"\n  ✅ RT-DETR-L model saved → {out_rtdetr}")
print(f"  ✅ {len(rows_rtdetr)} result rows ready (one per slice size)")

# Free GPU memory before next model
del model_rtdetr, results_rtdetr
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"  🧹 GPU cache cleared — "
          f"{torch.cuda.memory_reserved(0)/1e9:.1f} GB still reserved")

print("\n📌 Proceed to Cell 4 — Train YOLOv8m")


🏋️  Training RT-DETR-L  |  batch=4  |  device=cuda:0
   Architecture : Transformer-based (DETReg) — strong on large objects
  ⏱  Expected  : ~2.5 hr on T4 GPU | ~8+ hr on CPU
   Do NOT interrupt — transformer training is sensitive to early stop
Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=45.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr

## ⚡ Cell 4 — Train YOLOv8m + 3-Slice SAHI + Export NB2 Results
> **Expected time:** ~1.5 hr on T4 GPU · ~6+ hr on CPU  
> After training, exports `nb2_results.csv` (6 rows: 2 models × 3 slices).  
> Also prints the full 15-param before/after SAHI table.

In [4]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4 — TRAIN YOLOv8m + 3-SLICE SAHI + EXPORT ALL NB2 RESULTS
# ═══════════════════════════════════════════════════════════════════════

V8M_BATCH = 4   # ← conservative — matches RT-DETR-L for fair comparison

print("=" * 65)
print(f"🏋️  Training YOLOv8m  |  batch={V8M_BATCH}  |  device={device}")
print("=" * 65)
print("   Architecture : CNN-based (anchor-free) — fast, strong baseline")
print(f"  ⏱  Expected  : ~1.5 hr on T4 GPU | ~6+ hr on CPU")

# Build tracked param row
tracked_v8m = {k: COMMON_TRAIN_PARAMS[k] for k in TRACKED_PARAM_KEYS}
tracked_v8m["batch"] = V8M_BATCH

# Build training call
train_call_v8m = {k: v for k, v in COMMON_TRAIN_PARAMS.items()
                  if k not in ("conf", "iou")}
train_call_v8m.update({
    "batch":   V8M_BATCH,
    "project": f"{WORK_DIR}/runs",
    "name":    "YOLOv8m",
    "device":  0 if torch.cuda.is_available() else "cpu",
})

model_v8m   = YOLO("yolov8m.pt")
t_start     = time.time()
results_v8m = model_v8m.train(**train_call_v8m)
t_v8m       = time.time() - t_start
print(f"\n  ⏱  YOLOv8m training complete: {t_v8m/3600:.2f} hr")

# ── 3-Slice SAHI ablation ────────────────────────────────────────────
best_v8m  = os.path.join(results_v8m.save_dir, "weights", "best.pt")
rows_v8m  = []

print("\n  🔪 Running 3-slice SAHI ablation on YOLOv8m...")
print(f"  {'Slice':>8}  {'Standard':>10}  {'SAHI':>10}  {'Gain %':>8}")
print("  " + "─"*44)

for s_size in SAHI_SIZES:
    sahi_info = run_sahi_ablation(model_v8m, best_v8m, TEST_IMG_DIR, s_size)
    print(f"  {s_size:>6}px  {sahi_info['std_avg_det']:>10.2f}  "
          f"{sahi_info['sahi_avg_det']:>10.2f}  +{sahi_info['sahi_gain_pct']:>6.1f}%")
    row, _ = extract_metrics(results_v8m, "YOLOv8m", t_v8m,
                             sahi_info, V8M_BATCH, tracked_v8m)
    rows_v8m.append(row)

out_v8m = f"{SAVE_DIR}/YOLOv8m_best.pt"
shutil.copy(best_v8m, out_v8m)
print(f"\n  ✅ YOLOv8m model saved → {out_v8m}")

del model_v8m, results_v8m
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ── Save NB2 results CSV ─────────────────────────────────────────────
df_nb2  = pd.DataFrame(rows_rtdetr + rows_v8m)
NB2_CSV = f"{SAVE_DIR}/nb2_results.csv"
df_nb2.to_csv(NB2_CSV, index=False)

# ── Print the 15-param before/after SAHI table ───────────────────────
sahi_context = {
    "epochs":        "Train only — SAHI is an inference-time technique",
    "imgsz":         "Base res 640 → SAHI tiles into 480/320/160px slices",
    "batch":         "Train only — SAHI processes 1 image at a time",
    "patience":      "Train only — no effect on SAHI inference",
    "warmup_epochs": "Train only — no effect on SAHI inference",
    "mosaic":        "Train only — no effect on SAHI inference",
    "mixup":         "Train only — no effect on SAHI inference",
    "copy_paste":    "Train only — no effect on SAHI inference",
    "degrees":       "Train only — no effect on SAHI inference",
    "translate":     "Train only — no effect on SAHI inference",
    "scale":         "Train only — no effect on SAHI inference",
    "flipud":        "Train only — no effect on SAHI inference",
    "fliplr":        "Train only — no effect on SAHI inference",
    "hsv_h":         "Train only — no effect on SAHI inference",
    "hsv_s":         "Train only — no effect on SAHI inference",
}
print("\n" + "="*70)
print("15 HYPERPARAMETERS — BEFORE vs AFTER SAHI (NB2)")
print("="*70)
print(f"  {'ID':<5} {'Parameter':<18} {'Value':<10} {'SAHI context'}")
print("  " + "─"*74)
for i, k in enumerate(TRACKED_PARAM_KEYS, 1):
    print(f"  P{i:02d}   {k:<18} {str(COMMON_TRAIN_PARAMS[k]):<10} {sahi_context[k]}")
print("\n  SAHI-specific parameters (inference only — not in the 15):")
print(f"  SP01  sahi_slice_sizes = {SAHI_SIZES}  (3 sizes tested per model)")
print(f"  SP02  sahi_overlap     = {SAHI_OVERLAP}")
print(f"  SP03  sahi_conf        = {SAHI_CONF}")
print(f"  SP04  sahi_n_imgs      = {SAHI_N_IMGS}")

# ── NB2 results summary ──────────────────────────────────────────────
print("\n" + "="*70)
print("NB2 RESULTS SUMMARY (6 rows: 2 models × 3 slice sizes)")
print("="*70)
disp = ["Model","mAP50","mAP50_95","Precision","Recall","F1",
        "FPS","Params_M","std_avg_det","sahi_avg_det","sahi_gain_pct","sahi_slice"]
print(df_nb2[disp].to_string(index=False))

print(f"\n✅ NB2 results ({len(df_nb2)} rows) → {NB2_CSV}")
print("\n" + "="*65)
print("NEXT STEP — Upload outputs to a Kaggle Dataset")
print("="*65)
print("  1. Kaggle → Datasets → + New Dataset")
print("  2. Upload the full /kaggle/working/saved_models/ folder:")
print("     • RT-DETR-L_best.pt")
print("     • YOLOv8m_best.pt")
print("     • nb2_results.csv")
print("  3. Also upload NB1 outputs from Notebook 1:")
print("     • YOLO11n_best.pt  YOLO11s_best.pt  YOLO11m_best.pt")
print("     • nb1_results.csv")
print("  4. In NB3 Cell 1 → set INPUT_DATASET to your dataset path")
print("  5. Run Notebook 3 for full evaluation + figures + LaTeX table")


🏋️  Training YOLOv8m  |  batch=4  |  device=cuda:0
   Architecture : CNN-based (anchor-free) — fast, strong baseline
  ⏱  Expected  : ~1.5 hr on T4 GPU | ~6+ hr on CPU
Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=45.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yol